# Invoke Bedrock model for text generation using zero-shot prompt

> *This notebook should work well with the **`Python 3`** kernel in SageMaker Studio*

> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

## Introduction

In this notebook we show you how to use a LLM to generate an email response to a customer who provided negative feedback on the quality of customer service that they received from the support engineer. 

We will use Amazon Nova 2 Lite through the Converse API using the Boto3 SDK. 

The prompt used in this example is called a zero-shot prompt because we are not providing any examples of text alongside their classification other than the prompt.

**Note:** *This notebook can be run within or outside of AWS environment.*

#### Context
To demonstrate the text generation capability of Amazon Bedrock, we will explore the use of Boto3 client to communicate with Amazon Bedrock API. We will demonstrate different configurations available as well as how simple input can lead to desired outputs.

#### Pattern
We will simply provide the Amazon Bedrock API with an input consisting of a task, an instruction and an input for the model under the hood to generate an output without providing any additional example. The purpose here is to demonstrate how the powerful LLMs easily understand the task at hand and generate compelling outputs.

![](./images/bedrock.jpg)

#### Use case
To demonstrate the generation capability of models in Amazon Bedrock, let's take the use case of email generation.

#### Persona
You are Bob a Customer Service Manager at AnyCompany and some of your customers are not happy with the customer service and are providing negative feedbacks on the service provided by customer support engineers. Now, you would like to respond to those customers humbly aplogizing for the poor service and regain trust. You need the help of an LLM to generate a bulk of emails for you which are human friendly and personalized to the customer's sentiment from previous email correspondence.

#### Implementation
To fulfill this use case, in this notebook we will show how to generate an email with a thank you note based on the customer's previous email.We will use the Amazon Titan Text Large model using the Amazon Bedrock API with Boto3 client. 

## Setup

⚠️ ⚠️ ⚠️ Before running this notebook, ensure you've run the [Bedrock basics notebook](../00_Prerequisites/bedrock_basics.ipynb) notebook. ⚠️ ⚠️ ⚠️


In [1]:
import json
import os
import sys

import boto3
import botocore

boto3_bedrock = boto3.client('bedrock-runtime')
#- use this for with profile

## Generate text

Following on the use case explained above, let's prepare an input for  the Amazon Bedrock service to generate an email. With the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) the same message format works for every model provider, so no model-specific prompt formatting is needed.

In [2]:
# create the prompt
prompt_data = """
Command: Write an email from Bob, Customer Service Manager, to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer"""


Let's start by using the Amazon Nova 2 Lite model ([model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html)). It is called through its cross-Region inference profile `us.amazon.nova-2-lite-v1:0` and accepts the following parameters:
- `messages`: the conversation, here a single user turn with the prompt
- `inferenceConfig`: the parameters that the model takes into account while generating the output (`maxTokens`, `temperature`, `topP`)

In [3]:
messages = [{"role": "user", "content": [{"text": prompt_data}]}]
inference_config = {"maxTokens": 512, "topP": 0.95, "temperature": 0.1}

The Amazon Bedrock API provides you with an API `invoke_model` which accepts the following:
- `modelId`: This is the model ARN for the various foundation models available under Amazon Bedrock
- `accept`: The type of input request
- `contentType`: The content type of the output
- `body`: A json string consisting of the prompt and the configurations

Check [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids-arns.html) for Available text generation model Ids

#### Invoke the model with the Converse API

First, we explore how the model generates an output based on the prompt created earlier.

##### Complete Output Generation

In [4]:
modelId = "us.amazon.nova-2-lite-v1:0"
outputText = "\n"
try:

    response = boto3_bedrock.converse(modelId=modelId, messages=messages, inferenceConfig=inference_config)

    outputText = response["output"]["message"]["content"][0]["text"]

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error

In [5]:
print(outputText)

### Subject: We Apologize and Appreciate Your Feedback – Immediate Follow-Up Attached  

Dear John Doe,

I hope this message finds you well.

My name is **Bob**, and I’m the **Customer Service Manager** here at **[Your Company Name]**. I recently reviewed the feedback you shared regarding your recent interaction with one of our customer support engineers, and I want to personally reach out to thank you for taking the time to share your concerns—and to offer our sincerest apologies for the experience you had.

At **[Your Company Name]**, we strive to deliver exceptional customer support with every interaction, and clearly, we fell short in your case. Your feedback is incredibly valuable to us, and I want to assure you that we are taking immediate action to address the issues you raised.

### What We’re Doing:
1. **Review and Analysis**: We have already begun a thorough review of the support case you were assigned to, including reviewing call logs, chat transcripts, and internal notes.  

##### Streaming Output Generation
Above is an example email generated by the Amazon Nova 2 Lite model by understanding the input request and using its inherent understanding of the different modalities. This request to the API is synchronous and waits for the entire output to be generated by the model.

Bedrock also supports that the output can be streamed as it is generated by the model in form of chunks. Below is an example of invoking the model with streaming option. `invoke_model_with_response_stream` returns a `ResponseStream` which you can read from.

_You may want to enable scrolling on your output cell below:_ 

In [6]:
output = []
try:

    response = boto3_bedrock.converse_stream(modelId=modelId, messages=messages, inferenceConfig=inference_config)
    stream = response.get("stream")

    i = 1
    if stream:
        for event in stream:
            if "contentBlockDelta" in event:
                text = event["contentBlockDelta"]["delta"]["text"]
                output.append(text)
                print(f'\t\t\x1b[31m**Chunk {i}**\x1b[0m\n{text}\n')
                i += 1

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error

		**Chunk 1**
###

		**Chunk 2**
 Subject



		**Chunk 3**
: We

		**Chunk 4**
 Apolog

		**Chunk 5**
ize and

		**Chunk 6**
 App

		**Chunk 7**
reciate Your Feedback

		**Chunk 8**
 –

		**Chunk 9**
 Immediate

		**Chunk 10**
 Follow

		**Chunk 11**
-Up Attached

		**Chunk 12**
  

Dear

		**Chunk 13**
 John Doe

		**Chunk 14**
,

I hope this message finds

		**Chunk 15**
 you well.



		**Chunk 16**
My

		**Chunk 17**
 name is **

		**Chunk 18**
Bob**, and I’m

		**Chunk 19**
 the **Customer Service Manager**

		**Chunk 20**
 here

		**Chunk 21**
 at **[

		**Chunk 22**
Your Company Name]**. I



		**Chunk 23**
 recently

		**Chunk 24**
 reviewed

		**Chunk 25**
 the

		**Chunk 26**
 feedback you shared

		**Chunk 27**
 regarding

		**Chunk 28**
 your

		**Chunk 29**
 recent interaction

		**Chunk 30**
 with one

		**Chunk 31**
 of our customer

		**Chunk 32**
 support engineers

		**Chunk 33**
, and I want

		**Chunk 34**
 to personally

		**Chunk 35**
 reach

		**Chunk 36**
 out to

		**Chunk 37**
 thank

		**Chunk 38**
 you for taking the time to

		**Chunk 39**
 share your concerns—and

		**Chunk 40**
 to offer

		**Chunk 41**
 my sincere apologies for

		**Chunk 42**
 the experience



		**Chunk 43**
 you

		**Chunk 44**
 had

		**Chunk 45**
.

At

		**Chunk 46**
 **[Your Company Name]

		**Chunk 47**
**, we strive

		**Chunk 48**
 to deliver exceptional

		**Chunk 49**
 service

		**Chunk 50**
 to

		**Chunk 51**
 every customer,

		**Chunk 52**
 and it

		**Chunk 53**
’s clear that we fell

		**Chunk 54**
 short in

		**Chunk 55**
 your case

		**Chunk 56**
. Your

		**Chunk 57**
 feedback is incredibly

		**Chunk 58**
 valuable to

		**Chunk 59**
 us, and I want to

		**Chunk 60**
 assure you that we are

		**Chunk 61**
 taking immediate

		**Chunk 62**
 action



		**Chunk 63**
 to address the

		**Chunk 64**
 issues you raised

		**Chunk 65**
.

### What

		**Chunk 66**
 We’re

		**Chunk 67**
 Doing:
1. **Review

		**Chunk 68**
 and

		**Chunk 69**
 Analysis

		**Chunk 70**
**: We have

		**Chunk 71**
 flagged

		**Chunk 72**
 the

		**Chunk 73**
 specific

		**Chunk 74**
 interaction and

		**Chunk 75**
 are

		**Chunk 76**
 conducting a thorough review of

		**Chunk 77**
 the conversation

		**Chunk 78**
 and

		**Chunk 79**
 overall

		**Chunk 80**
 handling

		**Chunk 81**
 of your case

		**Chunk 82**
.
2. **Training



		**Chunk 83**
 &

		**Chunk 84**
 Coaching

		**Chunk 85**
**: The

		**Chunk 86**
 support

		**Chunk 87**
 engineer involved

		**Chunk 88**
 will receive

		**Chunk 89**
 additional

		**Chunk 90**
 training

		**Chunk 91**
 focused

		**Chunk 92**
 on communication

		**Chunk 93**
, problem resolution

		**Chunk 94**
, and empathy

		**Chunk 95**
—

		**Chunk 96**
areas we

		**Chunk 97**
 recognize

		**Chunk 98**
 need

		**Chunk 99**
 reinforcement

		**Chunk 100**
.
3. **Process

		**Chunk 101**
 Improvement**: We are

		**Chunk 102**
 also reviewing our



		**Chunk 103**
 internal

		**Chunk 104**
 support protocols

		**Chunk 105**
 to

		**Chunk 106**
 prevent

		**Chunk 107**
 similar experiences

		**Chunk 108**
 for

		**Chunk 109**
 other

		**Chunk 110**
 customers.

### Our

		**Chunk 111**
 Commitment

		**Chunk 112**
 to You:
As

		**Chunk 113**
 a direct

		**Chunk 114**
 follow

		**Chunk 115**
-up, I

		**Chunk 116**
 have attached a **personal

		**Chunk 117**
 action

		**Chunk 118**
 plan** outlining the steps we

		**Chunk 119**
 are taking to ensure

		**Chunk 120**
 your

		**Chunk 121**
 experience is

		**Chunk 122**
 resolved to



		**Chunk 123**
 your

		**Chunk 124**
 satisfaction.

		**Chunk 125**
 My

		**Chunk 126**
 team will

		**Chunk 127**
 be

		**Chunk 128**
 reaching out to you directly within

		**Chunk 129**
 the

		**Chunk 130**
 next **24 business

		**Chunk 131**
 hours** to discuss

		**Chunk 132**
 your concerns in

		**Chunk 133**
 more

		**Chunk 134**
 detail and work

		**Chunk 135**
 collabor

		**Chunk 136**
atively toward

		**Chunk 137**
 a resolution

		**Chunk 138**
.



		**Chunk 139**
We value

		**Chunk 140**
 your trust and partnership

		**Chunk 141**
, John

		**Chunk 142**
, and we are



		**Chunk 143**
 committed to regaining

		**Chunk 144**
 your confidence

		**Chunk 145**
 through

		**Chunk 146**
 action

		**Chunk 147**
—not

		**Chunk 148**
 just words.



		**Chunk 149**
Please

		**Chunk 150**
 don

		**Chunk 151**
’t hesitate to reply

		**Chunk 152**
 to this

		**Chunk 153**
 email or contact me directly at

		**Chunk 154**
 **[Bob

		**Chunk 155**
’s Phone

		**Chunk 156**
 Number]** or

		**Chunk 157**
 **[Bob

		**Chunk 158**
’s Email]**

		**Chunk 159**
 if you have any further

		**Chunk 160**
 questions or would like to discuss

		**Chunk 161**
 this

		**Chunk 162**
 matter sooner



		**Chunk 163**
.

Thank you again

		**Chunk 164**
 for bringing

		**Chunk 165**
 this to

		**Chunk 166**
 our attention.

		**Chunk 167**
 We appreciate

		**Chunk 168**
 your honesty

		**Chunk 169**
 and give

		**Chunk 170**
 us the opportunity to improve

		**Chunk 171**
.

Warm

		**Chunk 172**
 regards,  
**

		**Chunk 173**
Bob Thompson

		**Chunk 174**
**  
Customer

		**Chunk 175**
 Service Manager  
**

		**Chunk 176**
[Your Company Name]**  


		**Chunk 177**
**[Phone

		**Chunk 178**
 Number]**  


		**Chunk 179**
**[Email Address

		**Chunk 180**
]**  
**[Company Website

		**Chunk 181**
]**  



		**Chunk 182**
---

*

		**Chunk 183**
Attachment: Action

		**Chunk 184**
_

		**Chunk 185**
Plan_John

		**Chunk 186**
_Doe

		**Chunk 187**
_Issue_

		**Chunk 188**
Resolution.pdf*



The above helps to quickly get output of the model and let the service complete it as you read. This assists in use-cases where there are longer pieces of text that you request the model to generate. You can later combine all the chunks generated to form the complete output and use it for your use-case

## Conclusion
You have now experimented with using `boto3` SDK which provides a vanilla exposure to Amazon Bedrock API. Using this API you have seen the use case of generating an email responding to a customer due to their negative feedback.

### Take aways
- Adapt this notebook to experiment with different models available through Amazon Bedrock such as Anthropic Claude and Meta Llama models.
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

## Thank You